# Worked Example: ACC–Frontal Connectivity

## Goal
Spectral connectivity on real feedback epochs with surrogate null. See 11_advanced_utility_interoperability.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from mne_connectivity import spectral_connectivity_epochs
from LFPAnalysis import load_lfp, oscillation_utils
from LFPAnalysis.config import LoadConfig

beh = pd.read_csv(Path('../../data/sample_beh.csv'))
epochs = load_lfp(
    LoadConfig(path=Path('../../data/sample_feedback_start-epo.fif'), file_format='mne', preload=True)
)
epochs.metadata = beh[['reward', 'rpe']]
epochs_sub = epochs.copy().pick(['racas1-racas2', 'rmolf5-rmolf6'])
con = spectral_connectivity_epochs(
    epochs_sub, method='coh', mode='multitaper', fmin=13, fmax=30, faverage=True, verbose=False
)
coh_mat = con.get_data(output='dense')[:, :, 0]
coh_value = float(coh_mat[1, 0])
print(f'Beta coherence ACC–frontal: {coh_value:.3f}')

## Plot connectivity value and surrogate distribution

In [ ]:
seed_data = epochs_sub.get_data()[:, 0, :]
surr = oscillation_utils.make_surrogate_arrays(
    seed_data, method='swap_epochs', n_shuffles=50, rng_seed=42, return_generator=False
)
target_mean = epochs_sub.get_data()[:, 1, :].mean(axis=0)
surr_coh = [
    float(np.corrcoef(surr[i].mean(axis=0), target_mean)[0, 1]) for i in range(min(20, len(surr)))
]
fig, axes = plt.subplots(1, 2, figsize=(10, 3))
im = axes[0].imshow(coh_mat, vmin=0, vmax=1, cmap='viridis')
axes[0].set_xticks([0, 1])
axes[0].set_yticks([0, 1])
axes[0].set_xticklabels(['ACC', 'frontal'])
axes[0].set_yticklabels(['ACC', 'frontal'])
axes[0].set_title('Beta coherence')
fig.colorbar(im, ax=axes[0], shrink=0.8)
axes[1].hist(surr_coh, bins=15, color='0.7', label='surrogate (approx)')
axes[1].axvline(coh_value, color='r', lw=2, label='observed')
axes[1].set(xlabel='Coupling proxy', title='Surrogate null (illustrative)')
axes[1].legend()
fig.tight_layout()
plt.show()

## Saving results

See chapter 15 (`15_saving_and_organizing_results`) for the recommended `results/` layout.

In [ ]:
# Uncomment to save. See chapter 15 for the recommended results/ layout.
# out = Path('../../results/worked-examples')
# out.mkdir(parents=True, exist_ok=True)
# np.savez(out / 'beta_coherence.npz', coh_mat=coh_mat, surr_coh=np.array(surr_coh))

## Next step

11_advanced_utility_interoperability for full connectivity API. Chapter 10b (`10b_first_time_resolved_stats`) for statistics.